# AI-Powered Spam & Phishing Detection System
This notebook builds a machine learning model to detect spam/phishing messages.

In [ ]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import requests


In [ ]:
# Download dataset
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
response = requests.get(url)
with open('sms.tsv', 'wb') as f:
    f.write(response.content)
print('Dataset downloaded')

In [ ]:
# Load dataset
df = pd.read_csv('sms.tsv', sep='\t', header=None, names=['label','text'])
df['label'] = df['label'].map({'spam':1, 'ham':0})
df.head()

In [ ]:
# Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df['text'] = df['text'].apply(clean_text)

In [ ]:
# Split data
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Vectorization
vectorizer = TfidfVectorizer(stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
# Train models
nb = MultinomialNB()
nb.fit(X_train_vec, y_train)

lr = LogisticRegression()
lr.fit(X_train_vec, y_train)

In [ ]:
# Evaluate
nb_pred = nb.predict(X_test_vec)
lr_pred = lr.predict(X_test_vec)

print('Naive Bayes Accuracy:', accuracy_score(y_test, nb_pred))
print('Logistic Regression Accuracy:', accuracy_score(y_test, lr_pred))
print(classification_report(y_test, lr_pred))

In [ ]:
# Confusion Matrix
ConfusionMatrixDisplay.from_estimator(lr, X_test_vec, y_test)
plt.show()

In [ ]:
# Prediction function
def predict_message(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])
    pred = lr.predict(vec)
    return '🚨 Spam' if pred[0]==1 else '✅ Safe'

print(predict_message('Win a free iPhone now'))
print(predict_message('Let us meet tomorrow'))